# Official MJD Transformer Training

Runs the standardized 3-tokenization × 2-position-encoding matrix for both MJD tasks. Data splitting, preprocessing, optimization, checkpointing, and evaluation come from the unchanged collaborator-provided `mjdbench` workflow.

In [1]:
from pathlib import Path
import json
import os
import sys
import time

import pandas as pd
import torch

# Support execution from either mjd_detector/notebooks or the local package.
candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path('/home/klz/Data/zeronu_benchmark/Transformer_Approach/mjd_detector'),
]
for candidate in candidate_roots:
    if (candidate / 'mjd_transformer').is_dir():
        candidate_string = str(candidate.resolve())
        if candidate_string not in sys.path:
            sys.path.insert(0, candidate_string)

try:
    from mjdbench import (
        DataConfig,
        TrainingConfig,
        evaluate_model,
        prepare_dataset,
        set_seed,
        train_model,
    )
except ModuleNotFoundError:
    from mjd_partner_workflow import (
        DataConfig,
        TrainingConfig,
        evaluate_model,
        prepare_dataset,
        set_seed,
        train_model,
    )

from mjd_transformer import MJDTransformer, TokenizationConfig

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Transformer package:', Path(sys.modules['mjd_transformer'].__file__).resolve())

PyTorch: 2.11.0+cu128
CUDA available: True
Transformer package: /home/klz/Data/zeronu_benchmark/Transformer_Approach/mjd_detector/mjd_transformer/__init__.py


In [2]:
DATA_ROOT = Path(os.environ.get(
    'MJD_BENCH_DATA',
    '/home/klz/Data/zeronu_benchmark/MJD',
))
PROJECT_ROOT = Path('/home/klz/Data/zeronu_benchmark/Transformer_Approach/mjd_detector')
OUTPUT_ROOT = PROJECT_ROOT / 'results' / 'transformer_official_v1'
SUMMARY_PATH = OUTPUT_ROOT / 'transformer_results.csv'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

data_config = DataConfig(
    data_root=DATA_ROOT,
    validation_fraction=0.10,
    baseline_samples=200,
    classification_amplitude_normalization=True,
    regression_waveform_scale=1.0,
    seed=42,
)
training_config = TrainingConfig()  # Shared collaborator defaults.
TASKS = ('classification', 'regression')

# Add a run ID here only when you intentionally want to skip it manually.
COMPLETED_OFFICIAL_RUNS = set()

print('Dataset:', DATA_ROOT)
print('Outputs:', OUTPUT_ROOT)
print('Data config:', data_config.to_dict())
print('Training config:', training_config.to_dict())

Dataset: /home/klz/Data/zeronu_benchmark/MJD
Outputs: /home/klz/Data/zeronu_benchmark/Transformer_Approach/mjd_detector/results/transformer_official_v1
Data config: {'data_root': '/home/klz/Data/zeronu_benchmark/MJD', 'validation_fraction': 0.1, 'baseline_samples': 200, 'classification_amplitude_normalization': True, 'regression_waveform_scale': 1.0, 'seed': 42}
Training config: {'batch_size': 64, 'epochs': 50, 'learning_rate': 0.0005, 'weight_decay': 0.0001, 'gradient_clip_norm': 1.0, 'early_stopping_patience': 5, 'seed': 42, 'num_workers': 0, 'device': 'auto', 'early_stopping_min_delta': 0.0, 'deterministic': False, 'use_amp': False, 'amp_precision': 'auto'}


In [3]:
EXPERIMENTS = [
    {'tokenization': tokenization, 'position_encoding': position_encoding}
    for tokenization in ('raw_patches', 'segment_summary', 'pulse_entities')
    for position_encoding in ('coordinate_mlp', 'fourier_coordinates')
]

def make_tokenization_config(name):
    if name == 'raw_patches':
        return TokenizationConfig(
            tokenization='raw_patches',
            patch_size=20,
        )
    if name == 'segment_summary':
        return TokenizationConfig(
            tokenization='segment_summary',
            token_count=500,
        )
    if name == 'pulse_entities':
        return TokenizationConfig(
            tokenization='pulse_entities',
            token_count=500,
            uniform_entity_fraction=0.5,
            entity_context_size=9,
        )
    raise ValueError(f'Unknown tokenization: {name}')

def run_is_complete(run_dir):
    required = ('run_config.json', 'best.pt', 'history.json', 'metrics.json', 'run_summary.json')
    return all((run_dir / name).is_file() for name in required)

def upsert_summary(row):
    if SUMMARY_PATH.is_file():
        table = pd.read_csv(SUMMARY_PATH)
        table = table.loc[table['run_id'] != row['run_id']]
    else:
        table = pd.DataFrame()
    table = pd.concat([table, pd.DataFrame([row])], ignore_index=True)
    table = table.sort_values(['task', 'tokenization', 'position_encoding'])
    table.to_csv(SUMMARY_PATH, index=False)

display(pd.DataFrame(EXPERIMENTS))

,tokenization,position_encoding
0,raw_patches,coordinate_mlp
1,raw_patches,fourier_coordinates
2,segment_summary,coordinate_mlp
3,segment_summary,fourier_coordinates
4,pulse_entities,coordinate_mlp
5,pulse_entities,fourier_coordinates


In [ ]:
for task in TASKS:
    print('\n' + '#' * 88)
    print(f'Preparing official {task} data once')
    print('#' * 88)
    task_data = prepare_dataset(
        task=task,
        data_config=data_config,
        batch_size=training_config.batch_size,
        num_workers=training_config.num_workers,
        limit_per_split=None,
    )
    print('Counts:', task_data.counts)

    for experiment in EXPERIMENTS:
        tokenization = experiment['tokenization']
        position_encoding = experiment['position_encoding']
        run_id = f'{task}__{tokenization}__{position_encoding}'
        run_dir = OUTPUT_ROOT / run_id

        if run_id in COMPLETED_OFFICIAL_RUNS or run_is_complete(run_dir):
            print(f'Skipping completed run: {run_id}')
            continue

        print('\n' + '=' * 88)
        print(run_id)
        print('=' * 88)
        run_dir.mkdir(parents=True, exist_ok=True)
        tokenization_config = make_tokenization_config(tokenization)

        # Seed before construction so initial weights are reproducible.
        set_seed(training_config.seed, training_config.deterministic)
        model = MJDTransformer(
            task=task,
            tokenization_config=tokenization_config,
            position_encoding=position_encoding,
            d_model=64,
            nhead=4,
            num_layers=2,
            dim_feedforward=256,
            dropout=0.1,
            num_frequencies=6,
        )
        parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
        run_config = {
            'run_id': run_id,
            'task': task,
            'data': data_config.to_dict(),
            'counts': task_data.counts,
            'training': training_config.to_dict(),
            'representation': model.config_dict(),
            'parameter_count': parameter_count,
        }
        (run_dir / 'run_config.json').write_text(
            json.dumps(run_config, indent=2), encoding='utf-8'
        )
        print('Trainable parameters:', f'{parameter_count:,}')

        training_start = time.perf_counter()
        history = train_model(
            model,
            task_data.train_loader,
            task_data.validation_loader,
            task=task,
            config=training_config,
            output_dir=run_dir,
        )
        training_seconds = time.perf_counter() - training_start

        evaluation_start = time.perf_counter()
        metrics = evaluate_model(
            model,
            task_data.test_loader,
            task=task,
            device=training_config.device,
            output_dir=run_dir,
            use_amp=training_config.use_amp,
            amp_precision=training_config.amp_precision,
        )
        evaluation_seconds = time.perf_counter() - evaluation_start
        checkpoint = torch.load(run_dir / 'best.pt', map_location='cpu', weights_only=False)
        epochs_completed = len(history)
        row = {
            'run_id': run_id,
            'task': task,
            'tokenization': tokenization,
            'position_encoding': position_encoding,
            'parameter_count': parameter_count,
            'train_events': task_data.counts['train'],
            'validation_events': task_data.counts['validation'],
            'test_events': task_data.counts['test'],
            'epochs_completed': epochs_completed,
            'best_epoch': int(checkpoint['epoch']),
            'best_validation_score': float(checkpoint['score']),
            'training_seconds': training_seconds,
            'minutes_per_epoch': training_seconds / max(epochs_completed, 1) / 60.0,
            'evaluation_seconds': evaluation_seconds,
            'test_auc': metrics.get('auc'),
            'test_accuracy': metrics.get('accuracy'),
            'test_mae_kev': metrics.get('mae_kev'),
            'test_rmse_kev': metrics.get('rmse_kev'),
            'test_bias_kev': metrics.get('bias_kev'),
            'test_loss': metrics.get('loss'),
        }
        (run_dir / 'run_summary.json').write_text(
            json.dumps(row, indent=2, allow_nan=True), encoding='utf-8'
        )
        upsert_summary(row)
        print(json.dumps(row, indent=2, allow_nan=True))


########################################################################################
Preparing official classification data once
########################################################################################
Counts: {'train': 936000, 'validation': 104000, 'test': 390000}
Skipping completed run: classification__raw_patches__coordinate_mlp
Skipping completed run: classification__raw_patches__fourier_coordinates
Skipping completed run: classification__segment_summary__coordinate_mlp

classification__segment_summary__fourier_coordinates
Trainable parameters: 113,409
Epoch 001/050 | train loss 0.40179 | validation loss 0.363854 | validation macro_auc 0.903608
Epoch 002/050 | train loss 0.335397 | validation loss 0.314473 | validation macro_auc 0.924952
Epoch 003/050 | train loss 0.308462 | validation loss 0.289901 | validation macro_auc 0.936722
Epoch 004/050 | train loss 0.291386 | validation loss 0.278063 | validation macro_auc 0.944293
Epoch 005/050 | train loss 0.277819 |

In [ ]:
results = pd.read_csv(SUMMARY_PATH) if SUMMARY_PATH.is_file() else pd.DataFrame()
print('Completed official runs:', len(results))
display(results)